In [2]:
from modelscope import snapshot_download
model_dir = snapshot_download('BAAI/bge-reranker-large', cache_dir='E:/projects/python/AI/models')

2026-03-06 16:55:01,338 - modelscope - INFO - Got 11 files, start to download ...


Processing 11 items:   0%|          | 0.00/11.0 [00:00<?, ?it/s]

2026-03-06 17:07:25,749 - modelscope - INFO - Download model 'BAAI/bge-reranker-large' successfully.


In [5]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
model.eval()
# 如果有 GPU，将模型移到 GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

pairs =[['what is panda?', 'The giant pandan is bear species endemic to Chain.']]
inputs = tokenizer(pairs, padding=True, truncation=True, return_tensors='pt')
inputs = {k: v.to(device) for k, v in inputs.items()}
scores = model(**inputs).logits.view(-1).float()
print(scores)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: E:/projects/python/AI/models\BAAI\bge-reranker-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tensor([1.6443], device='cuda:0', grad_fn=<ViewBackward0>)


In [ ]:
pairs = [
    ['what is panda?', 'The giant panda is a bear species endemic to China.'],  # 高相关
    ['what is panda?', 'Pandas are cute.'],                                     # 中等相关
    ['what is panda?', 'The Eiffel Tower is in Paris.']                        # 不相关
]
inputs = tokenizer(pairs, padding=True, truncation=True, return_tensors='pt')
inputs = {k: v.to(device) for k, v in inputs.items()}
scores = model(**inputs).logits.view(-1).float()
print(scores)